In [1]:
from torch.utils.cpp_extension import load

polynomial_cuda = load(
    name="polynomial_cuda",
    sources=["polynomial_cuda.cu"],
    verbose=True,
    extra_cuda_cflags=["-O3"]
)
'''
setup.py = production / packaging
load()   = experimentation / research
not using setup.py because of cuda and pytorch version mismatches, which can lead to compilation errors. 
load() is more flexible and can handle these mismatches better during development.
'''

Using /home/soham/.cache/torch_extensions/py312_cu121 as PyTorch extensions root...
Creating extension directory /home/soham/.cache/torch_extensions/py312_cu121/polynomial_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/soham/.cache/torch_extensions/py312_cu121/polynomial_cuda/build.ninja...
/home/soham/ml_env/lib/python3.12/site-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module polynomial_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/2] /usr/local/cuda-13.2/bin/nvcc --generate-dependencies-with-compile --dependency-output polynomial_cuda.cuda.o.d -DTORCH_EXTENSION_NAME=polynomial_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /home/soham/ml_env/lib/python3.12/site-packages/torch/include -isystem /home/soham/ml_env/lib/python3.12/site-packages/torch/include/torch/csrc/api/include -isystem /home/soham/ml_env/lib/python3.12/site-packages/torch/include/TH -isystem /home/soham/ml_env/lib/python3.12/site-packages/torch/include/THC -isystem /usr/local/cuda-13.2/include -isystem /usr/include/python3.12 -D_GLIBCXX_USE_CXX11_ABI=0 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,code=sm_75 --compiler-options '-fPIC' -O3 -std=c++17 -c /home/soham/CudaToPytor

Loading extension module polynomial_cuda...


'\nsetup.py = production / packaging\nload()   = experimentation / research\nnot using setup.py because of cuda and pytorch version mismatches, which can lead to compilation errors. \nload() is more flexible and can handle these mismatches better during development.\n'

In [2]:
import torch
import torch.nn as nn
import time

class CUDAPolynomialActivation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return polynomial_cuda.polynomial_activation(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Implement backward pass if needed
        raise NotImplementedError("Backward pass not implemented")

class PolynomialActivation(nn.Module):
    def __init__(self, implementation='pytorch'):
        super().__init__()
        self.implementation = implementation

    def forward(self, x):
        if self.implementation == 'pytorch':
            return x**2 + x + 1
        elif self.implementation == 'cuda':
            return CUDAPolynomialActivation.apply(x)
        else:
            raise ValueError(f"Unknown implementation: {self.implementation}")

# Benchmark function
def benchmark(func, x, name, num_runs=1000):
    start_time = time.time()
    for _ in range(num_runs):
        func(x)
    torch.cuda.synchronize()
    end_time = time.time()
    return f"{name}: {(end_time - start_time) / num_runs * 1000:.4f} ms"

# Main function to run benchmarks
def main():
    torch.manual_seed(0)
    x = torch.randn(1000000, device='cuda')

    pytorch_activation = PolynomialActivation(implementation='pytorch').cuda()
    cuda_activation = PolynomialActivation(implementation='cuda').cuda()

    out = cuda_activation.forward(x)
    print(out)

    pytorch_time = benchmark(pytorch_activation, x, "PyTorch built-in")
    cuda_time = benchmark(cuda_activation, x, "CUDA extension")

    print(pytorch_time)
    print(cuda_time)

if __name__ == "__main__":
    main()

tensor([0.9303, 0.7556, 5.3461,  ..., 0.9957, 1.7441, 1.1275], device='cuda:0')
PyTorch built-in: 0.1375 ms
CUDA extension: 0.0374 ms
